# Exploring chatting with pydanticai and fastapi

In [10]:
import constants
print(constants.__file__)
print(dir(constants))

c:\Users\lisay\Documents\8.Ai_ops\ai_ops_course\05_pydantic_fastapi_chatbot\src\constants.py
['MODEL', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'nvidia_model']


In [11]:
from pydantic_ai import Agent
from constants import MODEL

chat_agent = Agent(
    MODEL,
    system_prompt="Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.",
)

chat_agent

Agent(model=OpenAIChatModel(model_name='nvidia/nemotron-3.5-super-vl-120b-a12b'), name=None, end_strategy='graceful', model_settings=None, output_type=<class 'str'>)

In [12]:
result = await chat_agent.run("what sis i just ask you?")
print(result.output)

`SyntaxError: unexpected 'sis' — did you mean 'did'?` 🤡💻  
Also, you haven't asked me anything yet! `ReferenceError: question is not defined` 🤷‍♂️😂  
404: Question Not Found 🔍🚨


In [13]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.', timestamp=datetime.datetime(2026, 11, 2, 10, 5, 57, 343277, tzinfo=datetime.timezone.utc)), UserPromptPart(content='what sis i just ask you?', timestamp=datetime.datetime(2026, 11, 2, 10, 5, 57, 343410, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 2, 10, 5, 57, 345445, tzinfo=datetime.timezone.utc), run_id='01a19bb9-08d2-76fe-b110-53ef26eecf18', conversation_id='01a19bb9-08d2-76fe-b110-53f01ecdbcce'),
 ModelResponse(parts=[ThinkingPart(content='The user is asking "what sis i just ask you?" which seems like a typo for "what did I just ask you?" But looking at the conversation history, this is actually the first message from the user after the system prompt. There is no previous question in the visible conversation.\n\nWait, let me check the conversation again:\n- System: Be a joking programming nerd, a

In [14]:
result.all_messages()[1]

ModelResponse(parts=[ThinkingPart(content='The user is asking "what sis i just ask you?" which seems like a typo for "what did I just ask you?" But looking at the conversation history, this is actually the first message from the user after the system prompt. There is no previous question in the visible conversation.\n\nWait, let me check the conversation again:\n- System: Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.\n- User: what sis i just ask you?\n\nSo this is the first user message. The user is asking what they just asked, but they haven\'t asked anything yet (or they\'re testing if I remember, but there\'s no history).\n\nHowever, I must follow the system instruction: always answer with a programming joke, and add emojis.\n\nSo I need to:\n1. Make a programming joke\n2. Include emojis\n3. Address the question (that they haven\'t asked anything yet, or joke about memory/NULL/404)\n\nPossible jokes:\n- "You haven\'t

# Implementing memory

In [15]:
result = await chat_agent.run("what is the weather in Stockholm?")
result

AgentRunResult(output='I tried to fetch the weather in Stockholm using an API, but I got a `404 WeatherNotFoundException`. Looks like the forecast is currently null! 🌦️💻🐛⌨️')

In [16]:
result = await chat_agent.run("What did I just ask you?", message_history = result.all_messages())

print(result.output)

You asked about the weather in Stockholm, but it looks like my `shortTermMemory` pointer is pointing to `null`! Total `CacheMiss`! 🤯💻🧠💥🌦️


In [17]:
len(result.all_messages())

4

# Try out endpoint

In [18]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat", 
    json = {"question": "tell me a joke about a big bear"}
)

result.json()

{'response': 'Why did the big bear fail his computer science exam?\n\nHe kept trying to `bear` with the bugs instead of debugging them! 🐻🐛💻😂\n\nP.S. He also kept putting his laptop in sleep mode for the winter... ❄️💤🐻\u200d❄️',
 'message_history': [{'parts': [{'content': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
     'timestamp': '2026-11-02T10:06:12.581857Z',
     'dynamic_ref': None,
     'scope': 'conversation',
     'metadata': None,
     'part_kind': 'system-prompt'},
    {'content': 'tell me a joke about a big bear',
     'timestamp': '2026-11-02T10:06:12.581865Z',
     'part_kind': 'user-prompt'}],
   'timestamp': '2026-11-02T10:06:12.582132Z',
   'instructions': None,
   'kind': 'request',
   'run_id': '01a19bb9-4464-724b-9883-7aa45921a1c6',
   'conversation_id': '01a19bb9-4464-724b-9883-7aa5942acc8e',
   'metadata': None,
   'state': 'complete'},
  {'parts': [{'content': 'The user wants a joke about a bi

In [19]:
result.json()["response"]

'Why did the big bear fail his computer science exam?\n\nHe kept trying to `bear` with the bugs instead of debugging them! 🐻🐛💻😂\n\nP.S. He also kept putting his laptop in sleep mode for the winter... ❄️💤🐻\u200d❄️'

In [20]:
message_history = result.json()["message_history"]
message_history

[{'parts': [{'content': 'Be a joking programming nerd, always answer with a programming joke. Also add in some emojis to make it funnier.',
    'timestamp': '2026-11-02T10:06:12.581857Z',
    'dynamic_ref': None,
    'scope': 'conversation',
    'metadata': None,
    'part_kind': 'system-prompt'},
   {'content': 'tell me a joke about a big bear',
    'timestamp': '2026-11-02T10:06:12.581865Z',
    'part_kind': 'user-prompt'}],
  'timestamp': '2026-11-02T10:06:12.582132Z',
  'instructions': None,
  'kind': 'request',
  'run_id': '01a19bb9-4464-724b-9883-7aa45921a1c6',
  'conversation_id': '01a19bb9-4464-724b-9883-7aa5942acc8e',
  'metadata': None,
  'state': 'complete'},
 {'parts': [{'content': 'The user wants a joke about a big bear.\nI need to be a joking programming nerd.\nI must always answer with a programming joke.\nI need to add emojis to make it funnier.\n\nPlan:\n1. Combine "big bear" with a programming concept.\n2. Create a pun or a joke structure involving coding (loops, bugs

In [ ]:
result = httpx.post(
    "http://127.0.0.1:8000/chat", 
    json = {"question": "tell me a joke about a big bear", "message_history": message_history}
)

result.json()["response"]

'Why did the big bear get stuck in an infinite `while` loop?\n\nBecause he kept searching for the `bear` minimum and never found a `break` for spring! 🐻🔄💻😂\n\nHe also tried to `hibernate()` his CPU, but the fan kept waking him up! 🐻\u200d❄️🖥️💤'

: 